# Aula 05: Relacionamentos no Modelo Entidade-Relacionamento (MER)
**Disciplina:** Banco de Dados | **Prof. Me. Reginaldo Pereira Fernandes** | **IFCE Campus Tauá**
---
## 🎯 Objetivos de Aprendizagem
- Compreender a definição formal de **Relacionamentos** no MER de Peter Chen (1976).
- Identificar e modelar os **Graus de Relacionamento**: Unário (com papéis/roles), Binário e Ternário.
- Dominar as **Razões de Cardinalidade**: $1:1$, $1:N$ e $N:M$.
- Compreender as **Restrições de Participação**: Total (linha dupla) vs. Parcial (linha simples) e a notação $(min, max)$.
- Identificar **Atributos de Relacionamentos** (atributos descritivos da associação).
- Validar as regras conceituais em Python puro (sem comandos SQL).

## 1. Metamodelo Conceitual do MER em Python
Neste bloco, criamos classes Python puras para representar as restrições formais de Peter Chen.

In [ ]:
from enum import Enum
from typing import Any, Dict, List, Optional, Set, Tuple

class TipoCardinalidade(Enum):
    UM_PARA_UM = "1:1"
    UM_PARA_MUITOS = "1:N"
    MUITOS_PARA_MUITOS = "N:M"

class TipoParticipacao(Enum):
    TOTAL = "Total (Obrigatória / Linha Dupla)"
    PARCIAL = "Parcial (Opcional / Linha Simples)"

class InstanciaRelacionamento:
    def __init__(self, id_a: str, id_b: str, atributos: Optional[Dict[str, Any]] = None, papel_a: Optional[str] = None, papel_b: Optional[str] = None):
        self.id_a = id_a
        self.id_b = id_b
        self.atributos = atributos or {}
        self.papel_a = papel_a
        self.papel_b = papel_b

    def __repr__(self) -> str:
        papeis = f" ({self.papel_a} -> {self.papel_b})" if self.papel_a else ""
        attrs = f" | Dados: {self.atributos}" if self.atributos else ""
        return f"Ligação[{self.id_a} <---> {self.id_b}{papeis}{attrs}]"

class DefinicaoRelacionamento:
    def __init__(self, nome: str, entidade_a: str, entidade_b: str, cardinalidade: TipoCardinalidade, participacao_a: TipoParticipacao, participacao_b: TipoParticipacao, papeis: Optional[Tuple[str, str]] = None, atributos_descritivos: Optional[List[str]] = None):
        self.nome = nome
        self.entidade_a = entidade_a
        self.entidade_b = entidade_b
        self.cardinalidade = cardinalidade
        self.participacao_a = participacao_a
        self.participacao_b = participacao_b
        self.papeis = papeis
        self.atributos_descritivos = atributos_descritivos or []
        self.instancias: List[InstanciaRelacionamento] = []

    def adicionar_ligacao(self, id_a: str, id_b: str, atributos: Optional[Dict[str, Any]] = None):
        atributos = atributos or {}
        for attr in self.atributos_descritivos:
            if attr not in atributos:
                raise ValueError(f"Violação em {self.nome}: Atributo {attr} obrigatório.")
        if self.cardinalidade == TipoCardinalidade.UM_PARA_UM:
            for inst in self.instancias:
                if inst.id_a == id_a or inst.id_b == id_b:
                    raise ValueError(f"Violação de exclusividade 1:1 em {self.nome} para [{id_a}] ou [{id_b}].")
        elif self.cardinalidade == TipoCardinalidade.UM_PARA_MUITOS:
            for inst in self.instancias:
                if inst.id_b == id_b:
                    raise ValueError(f"Violação 1:N em {self.nome}: Entidade [{id_b}] já pertence a [{inst.id_a}].")
        elif self.cardinalidade == TipoCardinalidade.MUITOS_PARA_MUITOS:
            for inst in self.instancias:
                if inst.id_a == id_a and inst.id_b == id_b:
                    raise ValueError(f"Duplicidade em N:M: Par ({id_a}, {id_b}) já registrado.")
        p_a = self.papeis[0] if self.papeis else None
        p_b = self.papeis[1] if self.papeis else None
        nova = InstanciaRelacionamento(id_a, id_b, atributos, p_a, p_b)
        self.instancias.append(nova)
        print(f"✅ {nova}")

## 2. Testes de Cardinalidades com Minimundo Empresarial (Didático)
Testamos as regras de negócio de **DEPARTAMENTO**, **FUNCIONARIO** e **PROJETO**.

In [ ]:
# 1. Relacionamento 1:1 (GERENCIA)
rel_1_1 = DefinicaoRelacionamento("GERENCIA", "DEPARTAMENTO", "FUNCIONARIO", TipoCardinalidade.UM_PARA_UM, TipoParticipacao.TOTAL, TipoParticipacao.PARCIAL, atributos_descritivos=["data_posse"])
rel_1_1.adicionar_ligacao("Engenharia", "Ana Silva", {"data_posse": "2026-01-15"})

# 2. Relacionamento 1:N (LOTADO_EM)
rel_1_n = DefinicaoRelacionamento("LOTADO_EM", "DEPARTAMENTO", "FUNCIONARIO", TipoCardinalidade.UM_PARA_MUITOS, TipoParticipacao.PARCIAL, TipoParticipacao.TOTAL)
rel_1_n.adicionar_ligacao("Engenharia", "Ana Silva")
rel_1_n.adicionar_ligacao("Engenharia", "Carlos Souza")

# 3. Relacionamento N:M com Atributo Descritivo (TRABALHA_EM)
rel_n_m = DefinicaoRelacionamento("TRABALHA_EM", "FUNCIONARIO", "PROJETO", TipoCardinalidade.MUITOS_PARA_MUITOS, TipoParticipacao.PARCIAL, TipoParticipacao.PARCIAL, atributos_descritivos=["horas_semanais"])
rel_n_m.adicionar_ligacao("Ana Silva", "Sistema Web", {"horas_semanais": 20})
rel_n_m.adicionar_ligacao("Ana Silva", "App Mobile", {"horas_semanais": 15})
rel_n_m.adicionar_ligacao("Carlos Souza", "Sistema Web", {"horas_semanais": 30})

# 4. Auto-relacionamento Unário com Papéis (SUPERVISAO)
rel_supervisao = DefinicaoRelacionamento("SUPERVISAO", "FUNCIONARIO", "FUNCIONARIO", TipoCardinalidade.UM_PARA_MUITOS, TipoParticipacao.PARCIAL, TipoParticipacao.PARCIAL, papeis=("Supervisor", "Supervisionado"))
rel_supervisao.adicionar_ligacao("Ana Silva", "Carlos Souza")
rel_supervisao.adicionar_ligacao("Ana Silva", "Beatriz Lima")

## 3. Demonstração de Bloqueio Conceitual de Violações
Demonstramos como o modelo conceitual rejeita violações de regras de cardinalidade.

In [ ]:
print("Tentando alocar um segundo gerente em Engenharia (Violação 1:1):")
try:
    rel_1_1.adicionar_ligacao("Engenharia", "Carlos Souza", {"data_posse": "2026-05-10"})
except ValueError as e:
    print(f"[BLOQUEADO COM SUCESSO]: {e}")

print("
Tentando lotar Carlos Souza em dois departamentos ao mesmo tempo (Violação 1:N):")
try:
    rel_1_n.adicionar_ligacao("Marketing", "Carlos Souza")
except ValueError as e:
    print(f"[BLOQUEADO COM SUCESSO]: {e}")